# Experiment B: Per-Domain Accuracy Breakdown (EqRelBench Table 1)

**Purpose:** Produce the actual numbers for Table 1 in the EqRelBench paper, which currently shows only XX.X placeholders.

This notebook evaluates 3 models across 5 semantic domains and 3 logical properties, producing the full breakdown table. It also tests **paraphrase sensitivity** within each domain, which is EqRelBench's key controlled perturbation.

## What this fills in
- **Table 1:** Accuracy (%) by model × property × domain
- **Table 2:** Accuracy by paraphrase variant per domain
- **Figure:** Chain depth scaling per domain (does depth sensitivity vary by domain?)

## Models
Set `MODELS` below. The paper uses 3 models covering different capability levels.
Default: Phi-3-mini (3.8B), SmolLM3-3B (3B), and Qwen2.5-0.5B (0.5B) — all open-weight, Colab-runnable.

## NOTE
Run Experiment A first if you want consistency scores alongside accuracy.
This notebook focuses on **accuracy by domain**, which is EqRelBench's primary table.

In [ ]:
!git clone https://github.com/Atharvy700/SP-25.git 2>/dev/null || echo 'Already cloned'
%cd SP-25
!pip install transformers accelerate -q

In [ ]:
import json, random, gc, torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from transformers import AutoTokenizer, AutoModelForCausalLM

random.seed(42)
np.random.seed(42)

print('torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# ── CONFIG: set models to evaluate ───────────────────────────────────────────
# Add or remove models here. Each must be a HuggingFace causal LM.
# Run sequentially (load → evaluate → unload) to stay within GPU memory.

MODELS = [
    {'name': 'microsoft/Phi-3-mini-4k-instruct',  'label': 'Phi-3-mini (3.8B)'},
    {'name': 'HuggingFaceTB/SmolLM3-3B',          'label': 'SmolLM3 (3B)'},
    {'name': 'Qwen/Qwen2.5-0.5B-Instruct',        'label': 'Qwen2.5 (0.5B)'},
]

N_PER_CELL = 150   # examples per (domain, property) cell. Increase for final paper.
MAX_LENGTH = 384

print(f'Models to evaluate: {len(MODELS)}')
for m in MODELS:
    print(f'  {m["label"]}')

In [ ]:
# ── Entity name pool ──────────────────────────────────────────────────────────
def gen_names(n=50000, lo=4, hi=7, seed=42):
    rng = random.Random(seed)
    pool = set()
    while len(pool) < n:
        k = rng.randint(lo, hi)
        pool.add(''.join(rng.choices('ABCDEFGHIJKLMNOPQRSTUVWXYZ', k=k)))
    return list(pool)

NAMES = gen_names()
pool  = NAMES[:8000]
print(f'Name pool: {len(NAMES)}, using first {len(pool)} for sampling.')

In [ ]:
# ── Domain definitions ────────────────────────────────────────────────────────
# Each domain has:
#   - 3 paraphrase variants for the SAME logical relation (tests surface sensitivity)
#   - reflexivity question template
# Paraphrase variants share identical logical structure, different surface form.

DOMAINS = {
    'abstract': {
        'label': 'Abstract',
        'paraphrases': [
            # (premise_template, question_template)
            ('{A} equals {B}.',          'Does {A} equal {B}?'),
            ('{A} is equal to {B}.',     'Is {A} equal to {B}?'),
            ('{A} and {B} are equal.',   'Are {A} and {B} equal?'),
        ],
        'refl_q': 'Does {A} equal {A}?',
    },
    'kinship': {
        'label': 'Kinship',
        'paraphrases': [
            ('{A} is the same age as {B}.',            'Is {A} the same age as {B}?'),
            ('{A} and {B} were born in the same year.', 'Were {A} and {B} born in the same year?'),
            ('{A} shares a birthday year with {B}.',    'Does {A} share a birthday year with {B}?'),
        ],
        'refl_q': 'Is {A} the same age as {A}?',
    },
    'set_theory': {
        'label': 'Set Theory',
        'paraphrases': [
            ('Set {A} has the same cardinality as set {B}.',         'Does set {A} have the same cardinality as set {B}?'),
            ('Sets {A} and {B} contain the same number of elements.', 'Do sets {A} and {B} contain the same number of elements?'),
            ('Set {A} is equinumerous with set {B}.',                 'Is set {A} equinumerous with set {B}?'),
        ],
        'refl_q': 'Does set {A} have the same cardinality as set {A}?',
    },
    'object_identity': {
        'label': 'Object Identity',
        'paraphrases': [
            ('Object {A} is identical to object {B}.',   'Is object {A} identical to object {B}?'),
            ('Object {A} and object {B} are the same.',  'Are object {A} and object {B} the same?'),
            ('{A} and {B} refer to the same object.',    'Do {A} and {B} refer to the same object?'),
        ],
        'refl_q': 'Is object {A} identical to object {A}?',
    },
    'synonymy': {
        'label': 'Synonymy',
        'paraphrases': [
            ('The term {A} is synonymous with the term {B}.',  'Is the term {A} synonymous with the term {B}?'),
            ('Words {A} and {B} have the same meaning.',       'Do words {A} and {B} have the same meaning?'),
            ('{A} means the same thing as {B}.',               'Does {A} mean the same thing as {B}?'),
        ],
        'refl_q': 'Is the term {A} synonymous with the term {A}?',
    },
    'spatial': {
        'label': 'Spatial',
        'paraphrases': [
            ('Location {A} is at the same coordinates as location {B}.',  'Is location {A} at the same coordinates as location {B}?'),
            ('Place {A} and place {B} occupy the same position.',          'Do place {A} and place {B} occupy the same position?'),
            ('{A} and {B} are co-located.',                                'Are {A} and {B} co-located?'),
        ],
        'refl_q': 'Is location {A} at the same coordinates as location {A}?',
    },
}

INSTRUCTION = 'You are a logical reasoning assistant.\nAnswer with exactly one word.\n\n'
SUFFIX      = '\n\nAnswer only yes or no:'

print('Domains:', list(DOMAINS.keys()))

In [ ]:
# ── Prompt builders ───────────────────────────────────────────────────────────
rng = random.Random(42)

def make_refl_prompts(domain_key, n):
    """Reflexivity: Does X R X? Always yes."""
    d = DOMAINS[domain_key]
    entities = rng.sample(pool, n)
    prompts = []
    for X in entities:
        q = d['refl_q'].format(A=X)
        prompts.append(INSTRUCTION + q + SUFFIX)
    return prompts, ['yes'] * n


def make_sym_prompts(domain_key, n, paraphrase_idx=0):
    """Symmetry: premise AB, ask BA. All answers yes."""
    d   = DOMAINS[domain_key]
    prem_tmpl, q_tmpl = d['paraphrases'][paraphrase_idx]
    prompts, gold = [], []
    used = set()
    while len(prompts) < n:
        A, B = rng.sample(pool, 2)
        if (A, B) not in used:
            used.add((A, B))
            premise  = 'Suppose ' + prem_tmpl.format(A=A, B=B)
            question = q_tmpl.format(A=B, B=A)   # ask reversed
            prompts.append(INSTRUCTION + premise + ' ' + question + SUFFIX)
            gold.append('yes')
    return prompts, gold


def make_trans_prompts(domain_key, n, paraphrase_idx=0):
    """Transitivity: premise AB + BC, ask AC. All answers yes (positive examples only).
    We separately test negative examples."""
    d   = DOMAINS[domain_key]
    prem_tmpl, q_tmpl = d['paraphrases'][paraphrase_idx]
    prompts, gold = [], []
    used = set()
    while len(prompts) < n // 2:
        A, B, C = rng.sample(pool, 3)
        key = (A, B, C)
        if key not in used:
            used.add(key)
            p1 = 'Suppose ' + prem_tmpl.format(A=A, B=B)
            p2 = 'Suppose ' + prem_tmpl.format(A=B, B=C)
            q  = q_tmpl.format(A=A, B=C)
            prompts.append(INSTRUCTION + p1 + ' ' + p2 + ' ' + q + SUFFIX)
            gold.append('yes')
    # Negative examples: break one link
    neg_used = set()
    while len(prompts) < n:
        A, B, C = rng.sample(pool, 3)
        key = (A, B, C, 'neg')
        if key not in neg_used:
            neg_used.add(key)
            # Break at a random position using 'does not'
            break_prem_tmpl = prem_tmpl.replace('same', 'different').replace(
                'equal', 'not equal').replace('identical', 'not identical').replace(
                'synonymous', 'not synonymous').replace('co-located', 'not co-located').replace(
                'equinumerous', 'not equinumerous')
            if rng.random() < 0.5:
                p1 = 'Suppose ' + break_prem_tmpl.format(A=A, B=B)  # break first link
                p2 = 'Suppose ' + prem_tmpl.format(A=B, B=C)
            else:
                p1 = 'Suppose ' + prem_tmpl.format(A=A, B=B)
                p2 = 'Suppose ' + break_prem_tmpl.format(A=B, B=C)  # break second link
            q  = q_tmpl.format(A=A, B=C)
            prompts.append(INSTRUCTION + p1 + ' ' + p2 + ' ' + q + SUFFIX)
            gold.append('no')
    return prompts, gold


# Sanity-check one prompt per property
for dk in ['abstract', 'kinship']:
    rp, _ = make_refl_prompts(dk, 1)
    sp, _ = make_sym_prompts(dk, 1)
    tp, _ = make_trans_prompts(dk, 2)
    print(f'--- {dk} reflexivity ---')
    print(rp[0])
    print(f'--- {dk} symmetry ---')
    print(sp[0])
    print()

In [ ]:
# ── Model loader and evaluator ────────────────────────────────────────────────

def load_model(model_name):
    print(f'  Loading {model_name}...')
    tok = AutoTokenizer.from_pretrained(model_name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    mdl = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype=torch.float16, device_map='auto'
    )
    mdl.eval()
    yes_id = tok('yes', add_special_tokens=False).input_ids[0]
    no_id  = tok('no',  add_special_tokens=False).input_ids[0]
    print(f'  yes_id={yes_id}  no_id={no_id}')
    return mdl, tok, yes_id, no_id


def unload_model(mdl):
    del mdl
    gc.collect()
    torch.cuda.empty_cache()


@torch.inference_mode()
def predict_batch(mdl, tok, yes_id, no_id, prompts, batch_size=8):
    results = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        enc = tok(
            batch, return_tensors='pt', padding=True,
            truncation=True, max_length=MAX_LENGTH
        ).to(mdl.device)
        logits = mdl(**enc).logits[:, -1, :]
        for lg in logits:
            results.append('yes' if lg[yes_id] > lg[no_id] else 'no')
    return results


def evaluate_model_all_domains(mdl, tok, yes_id, no_id, model_label):
    """Evaluate a model across all domains and properties.
    Returns nested dict: {domain: {property: {acc, paraphrase_accs, ...}}}
    """
    results = {}
    for dk, dv in DOMAINS.items():
        print(f'    Domain: {dk}')
        dr = {}

        # ── Reflexivity ──────────────────────────────────────────────────────
        prompts, gold = make_refl_prompts(dk, N_PER_CELL)
        preds = predict_batch(mdl, tok, yes_id, no_id, prompts)
        dr['reflexivity'] = {
            'acc': float(np.mean([p == g for p, g in zip(preds, gold)])),
            'yes_rate': float(np.mean([p == 'yes' for p in preds])),
        }

        # ── Symmetry ─────────────────────────────────────────────────────────
        # Test all 3 paraphrase variants
        para_accs = []
        for pi in range(3):
            prompts, gold = make_sym_prompts(dk, N_PER_CELL, paraphrase_idx=pi)
            preds = predict_batch(mdl, tok, yes_id, no_id, prompts)
            para_accs.append(float(np.mean([p == g for p, g in zip(preds, gold)])))
        dr['symmetry'] = {
            'acc': float(np.mean(para_accs)),     # average across paraphrases
            'paraphrase_accs': para_accs,
            'paraphrase_variance': float(np.var(para_accs)),  # key diagnostic
        }

        # ── Transitivity ─────────────────────────────────────────────────────
        para_accs = []
        for pi in range(3):
            prompts, gold = make_trans_prompts(dk, N_PER_CELL, paraphrase_idx=pi)
            preds = predict_batch(mdl, tok, yes_id, no_id, prompts)
            para_accs.append(float(np.mean([p == g for p, g in zip(preds, gold)])))
        dr['transitivity'] = {
            'acc': float(np.mean(para_accs)),
            'paraphrase_accs': para_accs,
            'paraphrase_variance': float(np.var(para_accs)),
        }

        results[dk] = dr
        r = dr
        print(f'      refl={r["reflexivity"]["acc"]:.3f}  '
              f'sym={r["symmetry"]["acc"]:.3f}  '
              f'trans={r["transitivity"]["acc"]:.3f}')

    return results

print('Evaluation functions ready.')

In [ ]:
# ── RUN ALL MODELS ────────────────────────────────────────────────────────────
if not torch.cuda.is_available():
    raise RuntimeError('Enable GPU: Runtime → Change runtime type → GPU')

all_model_results = {}

for model_cfg in MODELS:
    mname  = model_cfg['name']
    mlabel = model_cfg['label']
    print(f'\n====== {mlabel} ======')

    mdl, tok, yes_id, no_id = load_model(mname)
    results = evaluate_model_all_domains(mdl, tok, yes_id, no_id, mlabel)
    all_model_results[mlabel] = results

    unload_model(mdl)
    print(f'  Done with {mlabel}.')

print('\nAll models evaluated.')

In [ ]:
# ── TABLE 1: Accuracy by model and property (EqRelBench main table) ───────────
# This is the table that currently shows XX.X in the paper.
props   = ['reflexivity', 'symmetry', 'transitivity']
models  = [m['label'] for m in MODELS]
domains = list(DOMAINS.keys())

print('TABLE 1: Accuracy (%) by Logical Property')
print('(Averaged across all domains — matches EqRelBench Table 1 format)')
print()
print(f'{"Model":<25} {"Reflexivity":>12} {"Symmetry":>10} {"Transitivity":>14}')
print('-' * 65)
for mlabel in models:
    row = []
    for prop in props:
        accs = [all_model_results[mlabel][dk][prop]['acc'] for dk in domains]
        row.append(np.mean(accs))
    print(f'{mlabel:<25} {row[0]*100:>12.1f} {row[1]*100:>10.1f} {row[2]*100:>14.1f}')
print()
print('Note: Each cell averages accuracy across all 6 semantic domains.')

In [ ]:
# ── TABLE 2: Full per-domain breakdown ────────────────────────────────────────
print('TABLE 2: Per-Domain Accuracy Breakdown')
print('This is the table the EqRelBench paper should also show.')
print()

for mlabel in models:
    print(f'\n--- {mlabel} ---')
    print(f'{"Domain":<20}', end='')
    for prop in props:
        print(f'  {prop.capitalize()[:10]:>10}', end='')
    print()
    print('-' * 52)
    for dk in domains:
        dlabel = DOMAINS[dk]['label']
        print(f'{dlabel:<20}', end='')
        for prop in props:
            acc = all_model_results[mlabel][dk][prop]['acc']
            print(f'  {acc*100:>10.1f}', end='')
        print()

In [ ]:
# ── TABLE 3: Paraphrase Sensitivity ──────────────────────────────────────────
# This is EqRelBench's key controlled perturbation. High variance = surface-form dependence.
print('TABLE 3: Paraphrase Sensitivity (Variance across 3 surface forms)')
print('High variance = model is sensitive to surface form, not logical structure')
print()

for mlabel in models:
    print(f'\n--- {mlabel} ---')
    print(f'{"Domain":<20} {"Sym Var":>10} {"Trans Var":>12} {"Sym Para-1":>12} {"Sym Para-2":>12} {"Sym Para-3":>12}')
    print('-' * 80)
    for dk in domains:
        dlabel = DOMAINS[dk]['label']
        sym_var   = all_model_results[mlabel][dk]['symmetry']['paraphrase_variance']
        trans_var = all_model_results[mlabel][dk]['transitivity']['paraphrase_variance']
        sym_accs  = all_model_results[mlabel][dk]['symmetry']['paraphrase_accs']
        print(f'{dlabel:<20} {sym_var:>10.4f} {trans_var:>12.4f}', end='')
        for a in sym_accs:
            print(f'  {a*100:>10.1f}', end='')
        print()
    print()
    # Overall paraphrase variance
    all_sym_vars = [all_model_results[mlabel][dk]['symmetry']['paraphrase_variance'] for dk in domains]
    print(f'  Mean symmetry paraphrase variance: {np.mean(all_sym_vars):.4f}')
    if np.mean(all_sym_vars) > 0.01:
        print('  FINDING: Model is sensitive to surface form (variance > 0.01)')
    else:
        print('  FINDING: Model is relatively stable across paraphrases')

In [ ]:
# ── FIGURE 1: Per-domain accuracy grouped bar chart ───────────────────────────
# The core visualization for the EqRelBench paper.

n_models  = len(models)
n_domains = len(domains)
props_to_plot = ['reflexivity', 'symmetry', 'transitivity']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle(
    'EqRelBench: Accuracy by Domain and Logical Property',
    fontsize=13, fontweight='bold'
)

COLORS = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0'][:n_models]
x      = np.arange(n_domains)
width  = 0.8 / n_models

for ax, prop in zip(axes, props_to_plot):
    for i, (mlabel, color) in enumerate(zip(models, COLORS)):
        accs = [all_model_results[mlabel][dk][prop]['acc'] * 100 for dk in domains]
        offset = (i - n_models/2 + 0.5) * width
        bars = ax.bar(x + offset, accs, width, label=mlabel, color=color, alpha=0.85)

    ax.axhline(50, color='red', linestyle=':', alpha=0.7, linewidth=1.2, label='Chance (50%)')
    ax.set_xticks(x)
    ax.set_xticklabels([DOMAINS[dk]['label'].replace(' ', '\n') for dk in domains], fontsize=8)
    ax.set_ylim(0, 115)
    ax.set_ylabel('Accuracy (%)', fontsize=10)
    ax.set_title(prop.capitalize(), fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('expB_domain_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: expB_domain_accuracy.png')

In [ ]:
# ── FIGURE 2: Paraphrase variance by domain ───────────────────────────────────
# Shows whether surface form matters more in some domains than others.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    'Paraphrase Sensitivity: Accuracy Variance Across 3 Surface Forms\n'
    'Higher variance = model is sensitive to phrasing, not logical structure',
    fontsize=12, fontweight='bold'
)

for ax, prop in zip(axes, ['symmetry', 'transitivity']):
    for i, (mlabel, color) in enumerate(zip(models, COLORS)):
        variances = [all_model_results[mlabel][dk][prop]['paraphrase_variance']
                     for dk in domains]
        offset = (i - n_models/2 + 0.5) * (0.8 / n_models)
        ax.bar(x + offset, variances, 0.8/n_models,
               label=mlabel, color=color, alpha=0.85)

    ax.axhline(0.01, color='orange', linestyle='--', alpha=0.8,
               linewidth=1.5, label='Threshold (0.01)')
    ax.set_xticks(x)
    ax.set_xticklabels([DOMAINS[dk]['label'].replace(' ', '\n') for dk in domains], fontsize=8)
    ax.set_ylabel('Variance across paraphrases', fontsize=10)
    ax.set_title(f'{prop.capitalize()} paraphrase sensitivity', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('expB_paraphrase_variance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: expB_paraphrase_variance.png')

In [ ]:
# ── FIGURE 3: Radar chart — each model's profile across domains ───────────────
# Best for showing which domains each model handles well.

from matplotlib.patches import FancyArrowPatch
import matplotlib.patheffects as pe

n_domains = len(domains)
angles = np.linspace(0, 2 * np.pi, n_domains, endpoint=False).tolist()
angles += angles[:1]   # close the polygon
domain_labels = [DOMAINS[dk]['label'] for dk in domains]

fig, axes = plt.subplots(1, 3, figsize=(16, 5),
                          subplot_kw=dict(polar=True))
fig.suptitle(
    'Domain Accuracy Profiles per Property — Radar Chart\n'
    'Phi-3-mini-4k-instruct (3.8B)',
    fontsize=12, fontweight='bold'
)

if len(models) > 0:
    mlabel = models[0]  # Plot first model only in radar; add more as needed
    for ax, prop in zip(axes, props_to_plot):
        vals = [all_model_results[mlabel][dk][prop]['acc'] for dk in domains]
        vals += vals[:1]
        ax.plot(angles, vals, 'o-', linewidth=2, color='#2196F3')
        ax.fill(angles, vals, alpha=0.2, color='#2196F3')
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(domain_labels, fontsize=8)
        ax.set_ylim(0, 1)
        ax.set_yticks([0.25, 0.5, 0.75, 1.0])
        ax.set_yticklabels(['25%', '50%', '75%', '100%'], fontsize=6)
        ax.axhline(0.5, color='red', linestyle=':', alpha=0.5, linewidth=1)
        ax.set_title(f'{prop.capitalize()}\n{mlabel}', fontsize=10, pad=15)

plt.tight_layout()
plt.savefig('expB_radar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: expB_radar.png')

In [ ]:
# ── KEY FINDINGS SUMMARY ──────────────────────────────────────────────────────
print('=' * 70)
print('KEY FINDINGS FOR EQRELBENCH PAPER')
print('=' * 70)

for mlabel in models:
    print(f'\n[{mlabel}]')

    # 1. Is difficulty ordering stable? (refl < sym < trans expected)
    avg_by_prop = {}
    for prop in props:
        accs = [all_model_results[mlabel][dk][prop]['acc'] for dk in domains]
        avg_by_prop[prop] = np.mean(accs)

    ordered = avg_by_prop['reflexivity'] <= avg_by_prop['symmetry'] <= avg_by_prop['transitivity']
    inv_ordered = avg_by_prop['reflexivity'] >= avg_by_prop['symmetry'] >= avg_by_prop['transitivity']
    print(f'  Avg reflexivity:   {avg_by_prop["reflexivity"]*100:.1f}%')
    print(f'  Avg symmetry:      {avg_by_prop["symmetry"]*100:.1f}%')
    print(f'  Avg transitivity:  {avg_by_prop["transitivity"]*100:.1f}%')
    if ordered:
        print('  -> Difficulty ordering: refl <= sym <= trans (as expected)')
    elif inv_ordered:
        print('  -> UNEXPECTED: transitivity easiest — check dataset balance')
    else:
        print('  -> Non-monotone difficulty ordering — domain effects may dominate')

    # 2. Is failure cross-domain consistent?
    for prop in props:
        accs = [all_model_results[mlabel][dk][prop]['acc'] for dk in domains]
        print(f'  {prop} domain std: {np.std(accs):.4f}', end='')
        if np.std(accs) < 0.05:
            print(' (stable across domains)')
        else:
            best_d  = domains[np.argmax(accs)]
            worst_d = domains[np.argmin(accs)]
            print(f' (best: {best_d}, worst: {worst_d})')

    # 3. Paraphrase sensitivity
    all_sym_vars = [all_model_results[mlabel][dk]['symmetry']['paraphrase_variance']
                    for dk in domains]
    print(f'  Mean symmetry paraphrase variance: {np.mean(all_sym_vars):.4f}')

In [ ]:
# ── Save results ──────────────────────────────────────────────────────────────
def to_native(obj):
    if isinstance(obj, dict):
        return {k: to_native(v) for k, v in obj.items()}
    if isinstance(obj, (np.floating, float)):
        return round(float(obj), 6)
    if isinstance(obj, (np.integer, int)):
        return int(obj)
    if isinstance(obj, list):
        return [to_native(x) for x in obj]
    return obj

out = {
    'models':       [m['label'] for m in MODELS],
    'domains':      list(DOMAINS.keys()),
    'n_per_cell':   N_PER_CELL,
    'results':      to_native(all_model_results),
}

with open('experiment_B_results.json', 'w') as f:
    json.dump(out, f, indent=2)

print('Results saved to experiment_B_results.json')
print()
print('Files produced:')
print('  expB_domain_accuracy.png   — Figure 1: grouped accuracy bars by domain')
print('  expB_paraphrase_variance.png — Figure 2: paraphrase sensitivity')
print('  expB_radar.png             — Figure 3: domain radar chart per model')
print('  experiment_B_results.json  — raw numbers for all tables')